In [7]:
## Model evaluation — test-set profit comparison

Load coordinated RL, myopic, and single-market rolling-intrinsic **daily** `profit.csv` files and merge on **delivery calendar day** (Europe/Berlin). The RL file defines which days are in scope.

**Choose the RL run** with `SELECTED_MODEL` (folder `output/coordinated_multi_market/logging/<id>/`). Training ranges for each id are in `evaluation/rl_model_registry.py`.

Run from the **repository root** or from `evaluation/` (paths adjusted in the next cell).

In [8]:
# --- only switch you need for coordinated RL outputs ---
SELECTED_MODEL = "0"  # "0" … "5"; see evaluation.rl_model_registry.RL_MODEL_REGISTRY

from pathlib import Path
import sys

_root = Path.cwd().resolve()
if _root.name == "evaluation":
    sys.path.insert(0, str(_root.parent))
else:
    sys.path.insert(0, str(_root))

import pandas as pd

from evaluation.merge_test_profits import build_test_comparison_df, default_paths_bs15
from evaluation.rl_model_registry import get_rl_model_spec, registry_as_dataframe

spec = get_rl_model_spec(SELECTED_MODEL)
print(spec.model_number, spec.describe_training())

paths = default_paths_bs15(SELECTED_MODEL)
from IPython.display import display

display(registry_as_dataframe())
paths

{'rl': '/Users/timflaschel/Documents/GitHub/BessBidder1/output/coordinated_multi_market/logging/0/rolling_intrinsic_intelligently_stacked_on_day_ahead_qh/bs15cr1rto0.86mc365mt10/profit.csv',
 'myopic': '/Users/timflaschel/Documents/GitHub/BessBidder1/output/myopic_multi_market/rolling_intrinsic_stacked_on_day_ahead_qh/bs15cr1rto0.86mc365mt10/profit.csv',
 'single_glob': 'output/single_market/rolling_intrinsic/ri_basic/qh/*/bs15cr1rto0.86mc365mt10/profit.csv',
 'single_file': '/Users/timflaschel/Documents/GitHub/BessBidder1/output/single_market/rolling_intrinsic/ri_basic/qh/2019/bs15cr1rto0.86mc365mt10/profit.csv'}

In [9]:
# Prefer glob: single-market RI is often stored per calendar year under qh/<year>/...
try:
    df_test = build_test_comparison_df(
        paths["rl"],
        paths["myopic"],
        single_market_glob=paths["single_glob"],
    )
except FileNotFoundError:
    df_test = build_test_comparison_df(
        paths["rl"],
        paths["myopic"],
        single_market_profit_csv=paths["single_file"],
    )

df_test.head(10)

,delivery_day,day_rl,profit_rl_coordinated,profit_myopic,profit_single_market_ri
0,2021-04-01 00:00:00+02:00,2021-04-01 00:00:00+02:00,108.085996,92.200414,94.756532
1,2021-04-02 00:00:00+02:00,2021-04-02 00:00:00+02:00,164.820079,171.773142,170.710072
2,2021-04-03 00:00:00+02:00,2021-04-03 00:00:00+02:00,106.082374,108.628091,101.722099
3,2021-04-04 00:00:00+02:00,2021-04-04 00:00:00+02:00,138.618575,135.974138,146.519762
4,2021-04-05 00:00:00+02:00,2021-04-05 00:00:00+02:00,242.925466,246.913258,270.850620
5,2021-04-06 00:00:00+02:00,2021-04-06 00:00:00+02:00,297.350992,277.790859,260.867058
6,2021-04-07 00:00:00+02:00,2021-04-07 00:00:00+02:00,95.212557,137.939105,130.341463
7,2021-04-08 00:00:00+02:00,2021-04-08 00:00:00+02:00,125.831553,122.879300,150.934430
8,2021-04-09 00:00:00+02:00,2021-04-09 00:00:00+02:00,102.229895,94.549413,94.911422
9,2021-04-10 00:00:00+02:00,2021-04-10 00:00:00+02:00,126.908508,128.785421,136.493605


In [10]:
# Coverage check: baselines should have rows for each RL test day
summary = pd.DataFrame(
    {
        "rows": [len(df_test)],
        "myopic_missing": [df_test["profit_myopic"].isna().sum()],
        "single_market_missing": [df_test["profit_single_market_ri"].isna().sum()],
    }
)
summary

,rows,myopic_missing,single_market_missing
0,183,0,0


### Brute-force upper bound (per-day `summary_*.csv`)

For each test `delivery_day`, load `coordinated_market_upper_bound_analysis/results_merged/summary_YYYY-MM-DD.csv` and add **max / median / q75 / q90** of the `profit` column across all scenarios in that file.

For many repeated analyses you can later run `build_brute_force_long_cache()` once (see `evaluation/brute_force_summary_stats.py`).

In [ ]:
from evaluation.brute_force_summary_stats import attach_brute_force_stats, default_results_merged_dir

df_test_bf = attach_brute_force_stats(df_test, on_missing="warn")
df_test_bf.head()

In [11]:
df_test.describe()

,profit_rl_coordinated,profit_myopic,profit_single_market_ri
count,183.000000,183.000000,183.000000
mean,139.447068,140.191729,142.493344
std,63.729590,65.891204,62.641105
min,62.519791,54.627816,61.393795
25%,93.328858,92.254630,94.853304
50%,120.954435,120.253871,126.098533
75%,165.911189,165.646434,171.215533
max,370.463733,370.463733,349.226962


### SoC profiles from trade files (coordinated RL)

Trades live next to `profit.csv` in `trades/trades_YYYY-MM-DD.csv`. **DA** rows: `execution_time` hour == 13 (Berlin). **Actual** SoC uses all trades. SoC is integrated in **delivery product time** (96×15 min), same logic as your reference script (`soc_by_product_slots`).

In [ ]:
from evaluation.soc_profiles import rl_soc_plot_from_test_days

# Uses SELECTED_MODEL and test-period days from df_test (or df_test_bf)
fig, ax, mat_da, mat_all, s_da, s_all = rl_soc_plot_from_test_days(
    SELECTED_MODEL,
    df_test["delivery_day"],
    e_max_mwh=1.0,
    soc0_mwh=0.0,
    q_low=0.1,
    q_high=0.9,
)
fig.tight_layout()
fig